# Clean

In [ ]:
!pip install indobenchmark-toolkit
!pip install evaluate rouge_score

In [18]:
import sys
import types
import torch
import numpy as np
from transformers import AutoModelForSeq2SeqLM, GenerationConfig
from indobenchmark import IndoNLGTokenizer

# 1. FIX SYSTEM (OPERASI DARURAT)
import transformers.utils.generic
missing_flags = ["_is_jax", "_is_numpy", "_is_tensorflow", "_is_torch", "_is_torch_device"]
for flag in missing_flags:
    if not hasattr(transformers.utils.generic, flag):
        setattr(transformers.utils.generic, flag, False)

if not hasattr(transformers.utils, 'is_tf_available'):
    transformers.utils.is_tf_available = lambda: False

# 2. TOKENIZER PATCHER
def apply_tokenizer_patches(tokenizer):
    _original_pad = tokenizer.pad
    tokenizer.pad = lambda *args, **kwargs: _original_pad(*args, **{k: v for k, v in kwargs.items() if k != 'padding_side'})
    
    tokenizer.save_vocabulary = types.MethodType(lambda self, save_dir, prefix=None: (), tokenizer)
    
    _original_decode = tokenizer.decode
    def custom_decode(self, token_ids, skip_special_tokens=False, **kwargs):
        kwargs.pop('clean_up_tokenization_spaces', None)
        return _original_decode(token_ids, skip_special_tokens=skip_special_tokens, **kwargs)
    
    tokenizer.decode = types.MethodType(custom_decode, tokenizer)
    return tokenizer

# 3. LOAD MODEL & TOKENIZER
checkpoint_path = "/kaggle/input/models/muanaikhalifah/indobenchmarkindobart-v2-fine-tuned-on-indosum/pytorch/default/1/indosum-bart-summarization/checkpoint-38720"

raw_tokenizer = IndoNLGTokenizer.from_pretrained("indobenchmark/indobart-v2")
tokenizer = apply_tokenizer_patches(raw_tokenizer)
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

# 4. INFERENSIA YANG PRESISI (DENGAN FORCED BOS TOKEN)
def generate_summary_refined(text, model, tokenizer, device):
    id_id_token = tokenizer.convert_tokens_to_ids("[ind]")
    if id_id_token == tokenizer.unk_token_id:
        id_id_token = tokenizer.bos_token_id

    inputs = tokenizer(text, max_length=1024, truncation=True, return_tensors="pt").to(device)
    
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=128,
        min_length=30,
        num_beams=4,
        length_penalty=2.0,
        early_stopping=True,
        no_repeat_ngram_size=3,
        forced_bos_token_id=id_id_token,
        decoder_start_token_id=id_id_token
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Sistem siap di {device}. Gunakan 'generate_summary_refined' untuk evaluasi test_df.")

Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Sistem siap di cuda. Gunakan 'generate_summary_refined' untuk evaluasi test_df.


In [19]:
sample_idx = 0 # Ganti index sesuai keinginan
text_asli = test_df['text'].iloc[sample_idx]
target_asli = test_df['target'].iloc[sample_idx]

hasil_ai = generate_summary_refined(text_asli, model, tokenizer, device)

print(f"TARGET:\n{target_asli}\n")
print(f"MODEL AI:\n{hasil_ai}")

TARGET:
Satuan Pelaksana Program Indonesia Emas ( Satlak Prima ) mendapatkan jatah anggaran Rp 500 miliar untuk memenuhi kebutuhan atlet nasional selama 2017 . Jumlah itu dianggap masih jauh dari cukup untuk bisa memenuhi akomodasi , uang saku , vitamin dan kebutuhan lain yang menunjang performa atlet jelang turun di multievent . Ada 1000 atlet nasional baik itu senior maupun pratama yang mendapatkan SK ( Surat Keputusan ) pelatnas .

MODEL AI:
 satuan pelaksana program indonesia emas ( satlak prima ) mendapatkan jatah anggaran rp 500 miliar untuk memenuhi kebutuhan atlet nasional selama 2017 . jumlah itu dianggap masih jauh dari kata cukup untuk bisa memenuhi akomodasi , uang saku , vitamin dan kebutuhan lain untuk menunjang performa atlet jelang turun di multievent . saat ini ada 1000 atlet nasional baik itu senior maupun pratama yang mendapatkan sk ( surat keputusan ) pelatnas . sebab itu , sutjipto menyebut pihaknya telah mengusulkan kembali dana tambahan sebesar rp 90 miliar melal

# Dirty   

In [2]:
!pip install indobenchmark-toolkit
!pip install evaluate rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.4 MB/s eta 0:00:00


In [6]:
import sys
import transformers.utils
import transformers.utils.generic

missing_flags = ["_is_jax", "_is_numpy", "_is_tensorflow", "_is_torch", "_is_torch_device"]

for flag in missing_flags:
    if not hasattr(transformers.utils.generic, flag):
        setattr(transformers.utils.generic, flag, False)

if not hasattr(transformers.utils, 'is_tf_available'):
    def is_tf_available():
        try:
            import tensorflow as tf
            return True
        except ImportError:
            return Falsek
    transformers.utils.is_tf_available = is_tf_available

print("System Fixed")

System Fixed


In [7]:
import torch
import types
from transformers import AutoModelForSeq2SeqLM
from indobenchmark import IndoNLGTokenizer

def apply_tokenizer_patches(tokenizer):
    _original_pad = tokenizer.pad

    def custom_pad(*args, **kwargs):
        kwargs.pop('padding_side', None)
        return _original_pad(*args, **kwargs)

    tokenizer.pad = custom_pad

    def custom_save_vocabulary(self, save_directory, filename_prefix=None):
        return ()

    tokenizer.save_vocabulary = types.MethodType(custom_save_vocabulary, tokenizer)

    _original_decode = tokenizer.decode

    def custom_decode(self, token_ids, skip_special_tokens=False, clean_up_tokenization_spaces=None, **kwargs):
        return _original_decode(token_ids, skip_special_tokens=skip_special_tokens, **kwargs)

    tokenizer.decode = types.MethodType(custom_decode, tokenizer)

    return tokenizer

In [10]:
checkpoint_path = "/kaggle/input/models/muanaikhalifah/indobenchmarkindobart-v2-fine-tuned-on-indosum/pytorch/default/1/indosum-bart-summarization/checkpoint-38720"

from indobenchmark import IndoNLGTokenizer
from transformers import AutoModelForSeq2SeqLM

print("Memuat tokenizer dari Hugging Face Hub...")
raw_tokenizer = IndoNLGTokenizer.from_pretrained("indobenchmark/indobart-v2")
tokenizer = apply_tokenizer_patches(raw_tokenizer)

print("Memuat model dari checkpoint lokal...")
model = AutoModelForSeq2SeqLM.from_pretrained(checkpoint_path)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model dan Tokenizer siap digunakan di perangkat: {device}")

Memuat tokenizer dari Hugging Face Hub...


tokenizer_config.json:   0%|          | 0.00/303 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/932k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/279 [00:00<?, ?B/s]

Memuat model dari checkpoint lokal...


Loading weights:   0%|          | 0/264 [00:00<?, ?it/s]

Model dan Tokenizer siap digunakan di perangkat: cuda


In [11]:
def generate_summary(text, model, tokenizer, device, max_input=1024, max_target=128):
    inputs = tokenizer(text, max_length=max_input, truncation=True, return_tensors="pt").to(device)
    
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_length=max_target,
        num_beams=4,
        early_stopping=True,
        no_repeat_ngram_size=3
    )
    
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [16]:
base_path = "/kaggle/input/datasets/muanaikhalifah/indosum/"

manager = IndoSumManager(base_path)
dataset_processed, _, _ = manager.run_pipeline()
df_for_splitting = dataset_processed['train'].to_pandas()

preprocessor = DataPreprocessor(tokenizer)
raw_datasets = preprocessor.split_dataset(df_for_splitting)

test_df = raw_datasets['test'].to_pandas()

print(f"Test set berhasil dibangkitkan. Tersedia {len(test_df)} sampel untuk evaluasi.")

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/71353 [00:00<?, ? examples/s]

Map:   0%|          | 0/3743 [00:00<?, ? examples/s]

Map:   0%|          | 0/18774 [00:00<?, ? examples/s]

Map:   0%|          | 0/19359 [00:00<?, ? examples/s]

Map:   0%|          | 0/3743 [00:00<?, ? examples/s]

Map:   0%|          | 0/18774 [00:00<?, ? examples/s]

Test set berhasil dibangkitkan. Tersedia 1936 sampel untuk evaluasi.


In [17]:
import torch
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

sample_idx = 0 
sample_text = test_df['text'].iloc[sample_idx]
reference = test_df['target'].iloc[sample_idx]

id_id_token = tokenizer.convert_tokens_to_ids("[ind]")
if id_id_token == tokenizer.unk_token_id:
    id_id_token = tokenizer.bos_token_id

inputs = tokenizer(sample_text, return_tensors="pt", max_length=1024, truncation=True).to(device)

summary_ids = model.generate(
    inputs["input_ids"], 
    max_length=128, 
    min_length=20, 
    length_penalty=2.0, 
    num_beams=4, 
    early_stopping=True,
    forced_bos_token_id=id_id_token,
    decoder_start_token_id=id_id_token
)

generated_summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print("="*50)
print("HASIL EVALUASI MODEL")
print("="*50)
print(f"Teks Asli (Index {sample_idx}):\n{sample_text[:500]}...")
print("-" * 30)
print(f"Ringkasan Target (Ground Truth):\n{reference}")
print("-" * 30)
print(f"Ringkasan Model (AI Generated):\n{generated_summary}")
print("="*50)

HASIL EVALUASI MODEL
Teks Asli (Index 0):
Jakarta , CNN Indonesia - - Satuan Pelaksana Program Indonesia Emas ( Satlak Prima ) mendapatkan jatah anggaran Rp 500 miliar untuk memenuhi kebutuhan atlet nasional selama 2017 . Jumlah itu dianggap masih jauh dari kata cukup untuk bisa memenuhi akomodasi , uang saku , vitamin dan kebutuhan lain untuk menunjang performa atlet jelang turun di multievent . Ketua Satlak Prima Achmad Sutjipto mengatakan , saat ini ada 1000 atlet nasional baik itu senior maupun pratama yang mendapatkan SK ( Surat Ke...
------------------------------
Ringkasan Target (Ground Truth):
Satuan Pelaksana Program Indonesia Emas ( Satlak Prima ) mendapatkan jatah anggaran Rp 500 miliar untuk memenuhi kebutuhan atlet nasional selama 2017 . Jumlah itu dianggap masih jauh dari cukup untuk bisa memenuhi akomodasi , uang saku , vitamin dan kebutuhan lain yang menunjang performa atlet jelang turun di multievent . Ada 1000 atlet nasional baik itu senior maupun pratama yang mendap